In [ ]:
from tensorflow.keras.datasets import imdb

# 1. Get the word index dictionary mapping words to integers
word_index = imdb.get_word_index()

# 2. Reverse it so we map integers back to words
# We shift the index by 3 because Keras reserves indices 0, 1, 2 for padding, start, and unknown tokens.
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[1] = "[START]"
reverse_word_index[2] = "[OOV]" # Out of Vocabulary

# 3. Paste your exact array data here
sample_review_indices = [1, 14, 22, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]

# 4. Map and print the words
decoded_review = ' '.join([reverse_word_index.get(i, '?') for i in sample_review_indices])
print(decoded_review)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.layers import Input # Import the Input layer

# ==========================================
# SECTION 5: DATA PREPARATION FOR RNNs
# ==========================================
print("--- Section 5: Loading and Preprocessing Data ---")

# 1. Define configuration hyperparameters
max_features = 10000  # Only consider the top 10,000 most frequent words
max_len = 200         # Cut off reviews after 200 words (Timesteps)

# 2. Load the IMDB dataset
# The dataset comes pre-tokenized into word frequency ranking integers.
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)
print(f"Loaded {len(X_train)} training samples and {len(X_test)} test samples.")

# 3. Exploratory Data Analysis & Sanity Checks
print(f"Sample raw sequence entry (first 10 word indices): {X_train[0][:10]}")
sequence_lengths = [len(seq) for seq in X_train]
print(f"Average sequence length: {np.mean(sequence_lengths):.2f} words")
print(f"Max sequence length: {np.max(sequence_lengths)} words")
print(f"Class distribution (Target balance): {np.bincount(y_train)}")

# 4. Sequence Padding and Truncating
# We pad/truncate 'pre' to ensure the most emotionally significant words at the
# end of the movie review are preserved near the final recurrent hidden state.
X_train_pad = pad_sequences(X_train, maxlen=max_len, padding='pre', truncating='pre')
X_test_pad = pad_sequences(X_test, maxlen=max_len, padding='pre', truncating='pre')

# 5. Temporal-Safe Validation Split
# We hold out 20% of the training data specifically for monitoring overfitting.
val_split = 0.2
split_idx = int(len(X_train_pad) * (1 - val_split))

X_val = X_train_pad[split_idx:]
y_val = y_train[split_idx:]
X_train_final = X_train_pad[:split_idx]
y_train_final = y_train[:split_idx]

print(f"\nFinal Final Data Shapes:")
print(f"Train Shape: {X_train_final.shape} | Validation Shape: {X_val.shape}")

# ==========================================
# SECTION 6: BUILDING AND TRAINING RNN MODEL
# ==========================================
print("\n--- Section 6: Building Architecture ---")

# Hyperparameters for the model network
embedding_dim = 128
hidden_units = 64
learning_rate = 0.001
batch_size = 64
epochs = 10

def build_rnn_model(architecture_type='LSTM'):
    model = Sequential(name=f"{architecture_type}_Review_Classifier")

    # Explicitly define the input shape right at the start
    model.add(Input(shape=(max_len,), dtype='int32'))

    # Your regular layers continue below...
    model.add(Embedding(input_dim=max_features, output_dim=embedding_dim))
    model.add(SpatialDropout1D(0.2))

    if architecture_type == 'LSTM':
        model.add(LSTM(hidden_units, dropout=0.2, recurrent_dropout=0.2, return_sequences=False))
    elif architecture_type == 'GRU':
        model.add(GRU(hidden_units, dropout=0.2, recurrent_dropout=0.2, return_sequences=False))

    model.add(Dense(1, activation='sigmoid'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    return model

# Instantiate standard LSTM setup
model = build_rnn_model(architecture_type='LSTM')
model.summary()

# Setup Early Stopping to proactively mitigate overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

print("\n--- Training Model ---")
history = model.fit(
    X_train_final, y_train_final,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stop],
    verbose=1
)

# ==========================================
# SECTION 7: EVALUATION & SYSTEMATIC TUNING
# ==========================================
print("\n--- Section 7: Comprehensive Model Evaluation ---")

# 1. Extract and Plot Training Performance Metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'], label='Train Loss', color='teal')
ax1.plot(history.history['val_loss'], label='Val Loss', color='orange')
ax1.set_title('Cross-Entropy Loss Trajectory')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train Accuracy', color='teal')
ax2.plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
ax2.set_title('Accuracy Progression Curves')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
plt.show()

# 2. Evaluate performance on completely unseen test data
test_loss, test_acc = model.evaluate(X_test_pad, y_test, batch_size=batch_size, verbose=0)
print(f"\nFinal Test Set Evaluation Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f}")

# 3. Detailed classification reports for handling precision errors
y_pred_probs = model.predict(X_test_pad, batch_size=batch_size, verbose=0)
y_pred = (y_pred_probs > 0.5).astype("int32")

print("\nDetailed Performance Metrics Breakdown:")
print(classification_report(y_test, y_pred, target_names=['Negative Review', 'Positive Review']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))